# claim-qwt-r2-gate-nonregression — Tier A 对抗扫描（T4）

设计（已预注册，不得回调）：`experiments/qwt-r2-gate-nonregression/README.md` §2/§3。Phase 0 对齐 commit b5d27ce。

- 锚点（arXiv v4）：DeiT-T 4/4 RepQ-ViT 58.2 → +QwT 61.4（±0.5 容差，>1 停主实验）
- Phase 2：R²≤0 占比对照论文 "<5%"（E4）；置零模块与 PTQ 逐位一致断言
- Phase 3（Tier A）：DeiT-T 全扫描 calib_size {512,128,64,32} × gate {on,off} × seed {0,1,2}，PTQ 固化 seed0，逐种子只换补偿集
- 已知偏离（记录在案）：eval batch 64（官方 32，纯吞吐）；补偿回归复刻官方 L136-197 去 DDP gather；PTQ 固化（§1 解耦设计，非官方整脚本重跑）
- 会话自检点 5.5h

In [ ]:
import subprocess, sys, os, time
T0 = time.time()
BUDGET_H = 5.5
OUT = '/kaggle/working'
GIT_COMMIT = 'b5d27ce'
PIN_SHA = '8a927da86be764a85b7d552bcdfa937af8211f90'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm==0.9.16'], check=True)
import torch, timm, numpy as np
print('torch', torch.__version__, '| timm', timm.__version__, '| cuda', torch.version.cuda)
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'NO_GPU'

subprocess.run(['rm', '-rf', OUT + '/QwT'])
subprocess.run(['git', 'clone', '-q', 'https://github.com/wujx2001/QwT', OUT + '/QwT'], check=True)
subprocess.run(['git', '-C', OUT + '/QwT', 'checkout', '-q', PIN_SHA], check=True)
REPO_SHA = subprocess.run(['git', '-C', OUT + '/QwT', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('QwT HEAD:', REPO_SHA)

REPO = OUT + '/QwT/QwT-cls-RepQ-ViT'
sys.path.insert(0, REPO)
os.chdir(REPO)

freez = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout
open(OUT + '/pip-freeze.txt', 'w').write(freez)
ENV = {'gpu': torch.cuda.get_device_name(0), 'cuda': torch.version.cuda, 'torch': torch.__version__,
       'timm': timm.__version__, 'repo_sha': REPO_SHA, 'git_commit': GIT_COMMIT}
print('ENV:', ENV)

# ---- timm 0.9.16 × QwT vendor 兼容 shim（fire-0420 修复 2/2；本地已用同份 vendor 文件全链验证）----
# 背景：v3 于 vendor import 处崩——vendor 是 timm ~0.5-0.6 时代 API。四处兼容（不改 repo）：
import torch.nn as _nn
from copy import deepcopy as _deepcopy
# (1) 旧式 default_cfg 用 'hf_hub' 键，0.9.x PretrainedCfg 只认 'hf_hub_id'
import timm.models._registry as _reg
_orig_cfg_init = _reg.PretrainedCfg.__init__
def _cfg_hfhub_compat(self, **kwargs):
    if isinstance(kwargs.get('hf_hub'), str) and 'hf_hub_id' not in kwargs:
        kwargs['hf_hub_id'] = kwargs.pop('hf_hub')
    return _orig_cfg_init(self, **kwargs)
_reg.PretrainedCfg.__init__ = _cfg_hfhub_compat
# (2) overlay_external_default_cfg 在 0.9.16 被移除——按 0.6.13 历史实现补回
import timm.models.helpers as _th
if not hasattr(_th, 'overlay_external_default_cfg'):
    def overlay_external_default_cfg(default_cfg, kwargs):
        _e = kwargs.pop('external_default_cfg')
        _d = _deepcopy(default_cfg); _d.update(_e)
        return _d
    _th.overlay_external_default_cfg = overlay_external_default_cfg
# (3) 模块级 _init_vit_weights 被移除——按 0.6.13 形状移植（仅随机初始化路径，预训练加载后即覆盖）
import timm.models.vision_transformer as _tvit
if not hasattr(_tvit, '_init_vit_weights'):
    def _init_vit_weights(module, name='', head_bias=0., jax_impl=False):
        if jax_impl:
            if isinstance(module, _nn.Linear):
                _nn.init.xavier_uniform_(module.weight)
                if isinstance(module.bias, _nn.Parameter) and name and 'head' in name:
                    _nn.init.normal_(module.bias, std=1e-6)
        elif isinstance(module, _nn.Linear):
            _nn.init.trunc_normal_(module.weight, std=.02)
            if module.bias is not None:
                _nn.init.zeros_(module.bias)
            if name and 'head' in name:
                _nn.init.normal_(module.bias, std=1e-6)
        elif isinstance(module, _nn.LayerNorm):
            _nn.init.zeros_(module.bias); _nn.init.ones_(module.weight)
    _tvit._init_vit_weights = _init_vit_weights
# (4) build_model_with_cfg 弹 vendor 旧式键（default_cfg/pretrained_custom_load）；
#     vendor 按值导入，须重绑 _builder 与 helpers 两处（factory 无绑定，create_model 直调 create_fn=vendor 函数）
import timm.models._builder as _tb
_orig_bmc = _tb.build_model_with_cfg
def _bmc_compat(model_cls, variant, pretrained, *args, **kwargs):
    kwargs.pop('default_cfg', None)
    kwargs.pop('pretrained_custom_load', None)   # npz 自定义加载路径随之失效，权重走 HF（RESULTS 偏差标注）
    return _orig_bmc(model_cls, variant, pretrained, *args, **kwargs)
_tb.build_model_with_cfg = _bmc_compat
_th.build_model_with_cfg = _bmc_compat
print('timm compat shim applied: PretrainedCfg.hf_hub / overlay_external_default_cfg / _init_vit_weights / build_model_with_cfg legacy-kwargs')


In [ ]:
# 数据：train 类目录直用；val 运行时探测（类目录 / Annotations XML / devkit 三级 fallback）
DATA = '/kaggle/input/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC'
if not os.path.isdir(DATA):
    DATA = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if root.endswith('CLS-LOC') and os.path.isdir(os.path.join(root, 'train')) and os.path.isdir(os.path.join(root, 'val')):
            DATA = root; dirs[:] = []
        dirs[:] = [d for d in dirs if d not in ('train', 'val', 'test') and not d.startswith('.')]
assert DATA, 'DATA_NOT_FOUND: CLS-LOC 不在任何挂载路径'
print('DATA:', DATA)
val_dir = DATA + '/val'
subs = [d for d in os.listdir(val_dir) if os.path.isdir(os.path.join(val_dir, d))]
VAL_USED, VAL_SOURCE = val_dir, 'native_class_dirs'
if len(subs) < 1000:
    import xml.etree.ElementTree as ET
    ann = None
    ilsvrc = os.path.dirname(os.path.dirname(DATA))          # DATA=.../ILSVRC/Data/CLS-LOC -> .../ILSVRC
    cands = [os.path.join(ilsvrc, 'Annotations', 'CLS-LOC', 'val'),
             '/kaggle/input/imagenet-object-localization-challenge/ILSVRC/Annotations/CLS-LOC/val',
             '/kaggle/input/competitions/imagenet-object-localization-challenge/ILSVRC/Annotations/CLS-LOC/val']
    for cand in cands:
        if os.path.isdir(cand):
            ann = cand
            break
    if ann is None:
        for root, dirs, files in os.walk('/kaggle/input'):
            if root.endswith(os.path.join('Annotations', 'CLS-LOC', 'val')):
                ann = root; dirs[:] = []
                break
            dirs[:] = [d for d in dirs if d not in ('train', 'val', 'test') and not d.startswith('.')]
    assert ann is not None, 'DATA_VAL_UNSORTED: val 非 class 目录且无 Annotations/val（已含 DATA 根推导+walk 兜底仍未见）'
    VAL_USED, VAL_SOURCE = '/kaggle/working/val_sorted', 'xml_sorted_symlinks'
    os.makedirs(VAL_USED, exist_ok=True)
    n = 0
    for fn in sorted(os.listdir(ann)):
        if not fn.endswith('.xml'):
            continue
        stem = fn[:-4]
        root = ET.parse(os.path.join(ann, fn)).getroot()
        objs = root.findall('object')
        wnid = objs[0].findtext('name') if objs else None
        assert wnid, 'XML_NO_NAME: ' + fn
        img = None
        for ext in ('.JPEG', '.jpeg', '.jpg', '.png'):
            p = os.path.join(val_dir, stem + ext)
            if os.path.exists(p):
                img = p
                break
        assert img is not None, 'VAL_IMG_MISSING: ' + stem
        d = os.path.join(VAL_USED, wnid)
        os.makedirs(d, exist_ok=True)
        dst = os.path.join(d, stem + '.JPEG')
        if not os.path.exists(dst):
            os.symlink(img, dst)
        n += 1
    print('val sorted via XML:', n, 'images')
print('VAL_USED:', VAL_USED, '| source:', VAL_SOURCE)

from utils.utils import create_transform, create_loader, AverageMeter
from utils.build_model import build_model
from timm.data.dataset import ImageDataset
from timm.utils import accuracy

mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)
crop_pct = 0.875
dataset_train = ImageDataset(root=DATA + '/train', transform=create_transform('large_scale_train', mean, std, crop_pct))
dataset_eval = ImageDataset(root=VAL_USED, transform=create_transform('large_scale_test', mean, std, crop_pct))
print('train imgs:', len(dataset_train), '| val imgs:', len(dataset_eval))

loader_eval = create_loader(dataset_eval, batch_size=64, is_training=False, mean=mean, std=std,
                            num_workers=4, drop_last=False, re_prob=0.0)

def seed_all(s=0):
    import random
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed(s)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = True

In [ ]:
# PTQ（RepQ-ViT）seed0 固化 —— 复刻官方 qwt_vit_and_deit.py L280-421
from types import SimpleNamespace
from quant import quant_model, set_quant_state, QuantConv2d, QuantLinear, QuantMatMul

def enable_quant(submodel):
    for name, module in submodel.named_modules():
        if isinstance(module, (QuantConv2d, QuantLinear, QuantMatMul)):
            module.set_quant_state(True, True)

def disable_quant(submodel):
    for name, module in submodel.named_modules():
        if isinstance(module, (QuantConv2d, QuantLinear, QuantMatMul)):
            module.set_quant_state(False, False)
from torch.nn import Parameter

def make_train_loader():
    return create_loader(dataset_train, batch_size=32, is_training=True, mean=mean, std=std,
                         num_workers=4, drop_last=True, re_prob=0.0)

seed_all(0)
loader_train_0 = make_train_loader()
for data, _ in loader_train_0:
    calib_data = data.cuda()
    break
print('calib batch:', calib_data.shape)

args = SimpleNamespace(drop_path=0.0)
model = build_model('deit_tiny_patch16_224', args)
model.cuda().eval()
wq_params = {'n_bits': 4, 'channel_wise': True}
aq_params = {'n_bits': 4, 'channel_wise': False}
q_model = quant_model(model, input_quant_params=aq_params, weight_quant_params=wq_params)
q_model.cuda().eval()

set_quant_state(q_model, input_quant=True, weight_quant=True)
with torch.no_grad():
    _ = q_model(calib_data)

with torch.no_grad():
    module_dict = {}
    for name, module in q_model.blocks.named_modules():
        module_dict[name] = module
        idx = name.rfind('.')
        if idx == -1:
            idx = 0
        father_name = name[:idx]
        if father_name in module_dict:
            father_module = module_dict[father_name]
        else:
            raise RuntimeError(f'father module {father_name} not found')
        if 'norm1' in name or 'norm2' in name or 'norm' in name:
            if 'norm1' in name:
                next_module = father_module.attn.qkv
            elif 'norm2' in name:
                next_module = father_module.mlp.fc1
            else:
                next_module = father_module.reduction
            act_delta = next_module.input_quantizer.delta.reshape(-1)
            act_zero_point = next_module.input_quantizer.zero_point.reshape(-1)
            act_min = -act_zero_point * act_delta
            target_delta = torch.mean(act_delta)
            target_zero_point = torch.mean(act_zero_point)
            target_min = -target_zero_point * target_delta
            r = act_delta / target_delta
            b = act_min / r - target_min
            module.weight.data = module.weight.data / r
            module.bias.data = module.bias.data / r - b
            next_module.weight.data = next_module.weight.data * r
            if next_module.bias is not None:
                next_module.bias.data = next_module.bias.data + torch.mm(next_module.weight.data, b.reshape(-1, 1)).reshape(-1)
            else:
                next_module.bias = Parameter(torch.Tensor(next_module.out_features))
                next_module.bias.data = torch.mm(next_module.weight.data, b.reshape(-1, 1)).reshape(-1)
            next_module.input_quantizer.channel_wise = False
            next_module.input_quantizer.delta = target_delta
            next_module.input_quantizer.zero_point = target_zero_point
            next_module.weight_quantizer.inited = False

set_quant_state(q_model, input_quant=True, weight_quant=True)
with torch.no_grad():
    _ = q_model(calib_data)

@torch.no_grad()
def validate(m, loader):
    top1 = AverageMeter()
    m.eval()
    for inp, tgt in loader:
        inp, tgt = inp.cuda(), tgt.cuda()
        _, out = m(inp)
        a1, _ = accuracy(out, tgt, topk=(1, 5))
        top1.update(a1.item(), out.size(0))
    return top1.avg

TOP1_PTQ = validate(q_model, loader_eval)
print(f'PTQ(RepQ-ViT) top1 = {TOP1_PTQ:.2f} | anchor 58.2 | diff {TOP1_PTQ-58.2:+.2f}')

# Keep the quantizers and unwrapped blocks intact across compensation variants.
PTQ_MODEL = _deepcopy(q_model)
PTQ_STATE = {k: v.detach().cpu() for k, v in PTQ_MODEL.state_dict().items()}
torch.save(PTQ_STATE, OUT + '/ptq_deit_tiny_4_4_seed0.pt')
print('PTQ state saved,', len(PTQ_STATE), 'tensors')

In [ ]:
# 补偿机制（复刻官方 L54-197：CompensationBlock + 闭式回归 + 逐块链式），去 DDP gather
import torch.nn as nn
import hashlib

class CompensationBlock(nn.Module):
    def __init__(self, W, b, r2_score, block, linear_init=True, block_id=None):
        super().__init__()
        self.block = block
        self.lora_weight = nn.Parameter(torch.zeros((W.size(0), W.size(1))))
        self.lora_bias = nn.Parameter(torch.zeros(W.size(1)))
        if linear_init and (r2_score > 0):
            self.lora_weight.data.copy_(W)
            self.lora_bias.data.copy_(b)
            print(f'block {block_id} linear init (r2={r2_score:.3f})')
        else:
            nn.init.zeros_(self.lora_weight)
            nn.init.zeros_(self.lora_bias)
            print(f'block {block_id} ZERO init (r2={r2_score:.3f})')

    def forward(self, x):
        out = self.block(x)
        if self.training:
            out = out + x @ self.lora_weight.float() + self.lora_bias
        else:
            out = out + (x.half() @ self.lora_weight.half()).float() + self.lora_bias
        return out

def linear_regression_local(X, Y, block_id=0):
    X = X.reshape(-1, X.size(-1))
    X1 = torch.cat([X, torch.ones(size=[X.size(0),], device=X.device).reshape(-1, 1)], dim=-1)
    Y = Y.reshape(-1, Y.size(-1))
    W_all = torch.inverse(X1.t() @ X1) @ X1.t() @ Y
    W, b = W_all[:-1, :], W_all[-1, :]
    Y_pred = X @ W + b
    ss_tot = torch.sum((Y - Y.mean(dim=0)).pow(2))
    ss_res = torch.sum((Y - Y_pred).pow(2))
    r2 = 1 - ss_res / ss_tot
    return W, b, r2

def sha256_tensor(t):
    return hashlib.sha256(t.detach().cpu().numpy().tobytes()).hexdigest()

@torch.no_grad()
def run_variant(seed, size, gate):
    q_model = _deepcopy(PTQ_MODEL).eval()
    seed_all(seed)
    loader = make_train_loader()
    feature_batches = []
    remaining = size
    for images, _ in loader:
        images = images[:remaining]
        feature_batches.append(q_model.forward_before_blocks(images.cuda()))
        remaining -= len(images)
        if remaining == 0:
            break
    assert remaining == 0, 'incomplete calibration batch sequence'
    feats = torch.cat(feature_batches, dim=0)
    del feature_batches
    fhash = sha256_tensor(feats)
    output_previous = feats
    r2_list = []
    bit_results = {'n_checked': 0, 'n_bitidentical': 0, 'mismatches': []}
    for block_id in range(len(q_model.blocks)):
        fset = torch.utils.data.TensorDataset(output_previous.detach().cpu())
        floader = torch.utils.data.DataLoader(fset, batch_size=32, shuffle=False)
        block = q_model.blocks[block_id]
        fp_out, q_out, chain_in = [], [], []
        with torch.no_grad():
            for (t_out,) in floader:
                t_out = t_out.cuda()
                disable_quant(block)
                fp = block(t_out)
                enable_quant(block)
                qq = block(t_out)
                chain_in.append(t_out)
                fp_out.append(fp)
                q_out.append(qq)
        chain_in = torch.cat(chain_in, 0)
        fp_out = torch.cat(fp_out, 0)
        q_out = torch.cat(q_out, 0)
        assert torch.sum((output_previous - chain_in).abs()) < 1e-3
        W, b, r2 = linear_regression_local(chain_in, fp_out - q_out, block_id)
        r2_list.append(float(r2))
        r2_eff = float(r2) if gate == 'on' else 1.0
        new_block = CompensationBlock(W=W, b=b, r2_score=r2_eff, block=block, linear_init=True, block_id=block_id)
        q_model.blocks[block_id] = new_block
        new_block.cuda()
        if gate == 'on' and float(r2) <= 0:
            probe = chain_in[:8]
            o1 = block(probe)
            o2 = new_block.eval()(probe)
            new_block.training = True  # Match the official FP32 calibration path.
            same = torch.equal(o1, o2)
            bit_results['n_checked'] += 1
            bit_results['n_bitidentical'] += int(same)
            if not same:
                bit_results['mismatches'].append({'block': block_id, 'max_abs_diff': float((o1 - o2).abs().max())})
        output_previous = torch.zeros(0, device='cuda')
        with torch.no_grad():
            for (t_out,) in floader:
                t_out = t_out.cuda()
                enable_quant(new_block)
                output_previous = torch.cat([output_previous, new_block(t_out).detach()], 0)
    top1 = validate(q_model, loader_eval)
    return top1, r2_list, fhash, bit_results

print('compensation machinery ready')

In [ ]:
# Phase 1+2：锚点配置（seed0, 512, gate on）+ R² 统计 + 逐位一致结果
import json
METRICS = open(OUT + '/metrics.jsonl', 'a')

def log_metric(d):
    d['env'] = ENV
    d['elapsed_h'] = round((time.time() - T0) / 3600, 3)
    METRICS.write(json.dumps(d, ensure_ascii=False) + '\n')
    METRICS.flush()

top1_qwt, r2_list, fhash, bit_results = run_variant(0, 512, 'on')
diff_ptq, diff_qwt = TOP1_PTQ - 58.2, top1_qwt - 61.4
print(f'ANCHOR DeiT-T 4/4: PTQ {TOP1_PTQ:.2f} (want 58.2, diff {diff_ptq:+.2f}) | QwT {top1_qwt:.2f} (want 61.4, diff {diff_qwt:+.2f})')
ANCHOR_OK = abs(diff_ptq) <= 1.0 and abs(diff_qwt) <= 1.0
print('ANCHOR_OK:', ANCHOR_OK, '(±1.0 硬限；±0.5 内为理想)')

r2t = torch.tensor(r2_list)
n_zeroed = int((r2t <= 0).sum())
r2_stats = {'n_modules': len(r2_list), 'n_zeroed': n_zeroed,
            'zeroed_ratio': round(n_zeroed / len(r2_list), 4),
            'r2_min': round(float(r2t.min()), 4), 'r2_median': round(float(r2t.median()), 4),
            'r2_hist_deciles': [round(float(x), 3) for x in torch.quantile(r2t, torch.linspace(0, 1, 11))],
            'paper_claim': '<5% notably low (E4)'}
print('R2 stats:', r2_stats)
print('bit-identical:', bit_results)

log_metric({'type': 'anchor_phase12', 'model': 'deit_tiny', 'calib_seed': 0, 'calib_size': 512, 'gate': 'on',
            'top1_ptq': round(TOP1_PTQ, 3), 'top1_qwt': round(top1_qwt, 3),
            'delta': round(top1_qwt - TOP1_PTQ, 3), 'anchor_ok': ANCHOR_OK,
            'r2_stats': r2_stats, 'r2_by_block': r2_list, 'bit_identical': bit_results, 'feats_sha256': fhash,
            'val_source': VAL_SOURCE, 'eval_batch': 64})

assert ANCHOR_OK, 'ANCHOR_FAIL: 偏差 >1%，按 README Phase 1 停主实验（结果照实记录）'

In [ ]:
# Phase 3 对抗扫描（DeiT-T 全扫描：4 尺寸 × 2 门控 × 3 种子，锚点配置已计为 seed0/512/on）
order = []
for size in [512, 128, 64, 32]:
    for gate in ['on', 'off']:
        for sd in [0, 1, 2]:
            if size == 512 and gate == 'on' and sd == 0:
                continue
            order.append((size, gate, sd))
print('Phase3 configs:', len(order))

for size, gate, sd in order:
    if (time.time() - T0) / 3600 > BUDGET_H:
        print(f'BUDGET_STOP at {size}/{gate}/seed{sd} — 5.5h 自检点触发，收尾保存')
        break
    t = time.time()
    top1, r2l, fh, bitr = run_variant(sd, size, gate)
    d = round(top1 - TOP1_PTQ, 3)
    r2t = torch.tensor(r2l)
    log_metric({'type': 'phase3', 'model': 'deit_tiny', 'calib_seed': sd, 'calib_size': size, 'gate': gate,
                'top1_ptq': round(TOP1_PTQ, 3), 'top1_qwt': round(top1, 3), 'delta': d,
                'n_zeroed': int((r2t <= 0).sum()), 'r2_min': round(float(r2t.min()), 4), 'r2_by_block': r2l,
                'feats_sha256': fh, 'bit_identical': bitr, 'eval_batch': 64,
                'config_sec': round(time.time() - t, 1)})
    print(f'[{size}/{gate}/seed{sd}] top1={top1:.2f} Δ={d:+.3f} ({round(time.time()-t,1)}s)')

print('PHASE3_DONE_OR_BUDGET_STOPPED at elapsed_h =', round((time.time() - T0) / 3600, 2))

In [ ]:
# 收尾：汇总 + 三行结论
lines = [json.loads(l) for l in open(OUT + '/metrics.jsonl')]
p3 = [l for l in lines if l.get('type') == 'phase3']
anchor = [l for l in lines if l.get('type') == 'anchor_phase12']

deltas = [l['delta'] for l in p3]
tierA = {'n_configs': len(p3), 'delta_min': min(deltas) if deltas else None,
         'delta_median': sorted(deltas)[len(deltas)//2] if deltas else None,
         'n_negative': sum(1 for d in deltas if d <= -0.1),
         'by_gate': {g: {'n': sum(1 for l in p3 if l['gate'] == g),
                         'delta_min': min([l['delta'] for l in p3 if l['gate'] == g], default=None)}
                     for g in ['on', 'off']}}
json.dump({'env': ENV, 'anchor': anchor, 'tierA_summary': tierA, 'configs': p3},
          open(OUT + '/results_summary.json', 'w'), indent=1, ensure_ascii=False)

L1 = f'line1 anchor: PTQ {TOP1_PTQ:.2f}/58.2, QwT {top1_qwt:.2f}/61.4, anchor_ok={ANCHOR_OK}'
L2 = f'line2 phase2: n_zeroed={n_zeroed}/{len(r2_list)} ({r2_stats["zeroed_ratio"]} vs paper <5%), bit_identical {bit_results["n_bitidentical"]}/{bit_results["n_checked"]}'
L3 = f'line3 tierA: {tierA["n_configs"]} configs, delta_min={tierA["delta_min"]}, n_negative(<=-0.1)={tierA["n_negative"]}'
open(OUT + '/summary.txt', 'w').write('\n'.join([L1, L2, L3]))
print(L1)
print(L2)
print(L3)
print('DONE elapsed_h', round((time.time() - T0) / 3600, 2))